# Banking77 NLP Project

This project looks at two useful NLP tasks for customer support:

1. Intent classification, where a new ticket is assigned to one of 77 banking categories
2. Semantic search, where a new ticket is matched with similar historical tickets

I first build a simple TF-IDF and Logistic Regression baseline, then compare it with sentence embeddings and Logistic Regression.

The goal is not only to compare accuracy, but also to understand where each approach works well and where it struggles.


## 1. Import libraries

The main libraries used here are pandas, scikit-learn, and sentence-transformers.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer


## 2. Load the BANKING77 dataset

The dataset contains customer support questions grouped into 77 banking intents.

I use the original train and test files so the evaluation stays consistent.


In [ ]:
train_url = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv"
test_url = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv"

train_df = pd.read_csv(train_url)
test_df = pd.read_csv(test_url)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()


## 3. Quick data check

The input column is `text` and the target column is `category`.

Before modelling, I check the number of categories and how many training examples each category has.


In [ ]:
print("Number of categories:", train_df["category"].nunique())

train_df["category"].value_counts().head(10)


The training set is not perfectly balanced. Some categories have many more examples than others.

The test set contains 40 examples for each category, which makes class level evaluation easier to compare.


## 4. Prepare training and test data


In [ ]:
X_train = train_df["text"]
y_train = train_df["category"]

X_test = test_df["text"]
y_test = test_df["category"]


## 5. Model 1: TF-IDF and Logistic Regression

This is the baseline model.

TF-IDF converts text into numerical features based on important words and phrases. I use unigrams and bigrams so the model can learn features such as `refund`, `virtual card`, and `charged twice`.

Logistic Regression then learns which features are associated with each of the 77 categories.


In [ ]:
tfidf_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2
        )
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000
        )
    )
])

tfidf_pipeline.fit(X_train, y_train)

y_pred_tfidf = tfidf_pipeline.predict(X_test)


In [ ]:
tfidf_accuracy = accuracy_score(y_test, y_pred_tfidf)

print("TF-IDF Accuracy:", round(tfidf_accuracy, 4))


In my run, the TF-IDF model reached about 85.8% accuracy.

Accuracy is useful, but it does not show which categories are easy or difficult. So I also check precision, recall, and F1 score.


In [ ]:
print(classification_report(y_test, y_pred_tfidf))


## 6. Error analysis

One weak class in the TF-IDF model was `virtual_card_not_working`.

The model had high precision but low recall for this class. This means it was careful when predicting the class, but it missed many real examples.

To understand why, I inspect the wrong predictions.


In [ ]:
error_df = pd.DataFrame({
    "text": X_test.reset_index(drop=True),
    "actual": y_test.reset_index(drop=True),
    "predicted": y_pred_tfidf
})

virtual_errors = error_df[
    (error_df["actual"] == "virtual_card_not_working") &
    (error_df["predicted"] != "virtual_card_not_working")
]

virtual_errors["predicted"].value_counts()


In [ ]:
virtual_errors[
    ["text", "actual", "predicted"]
].head(10)


Most mistakes stayed inside the broader virtual card topic.

The model often predicted categories such as `getting_virtual_card` or `get_disposable_virtual_card`.

This suggests that TF-IDF was identifying the topic well, but sometimes struggled to understand the exact intent of the full sentence.


## 7. Inspect important TF-IDF features

Logistic Regression is useful because its learned coefficients can be inspected.

Here I look at the strongest features for `virtual_card_not_working`.


In [ ]:
tfidf = tfidf_pipeline.named_steps["tfidf"]
lr_model = tfidf_pipeline.named_steps["model"]

feature_names = tfidf.get_feature_names_out()

class_name = "virtual_card_not_working"
class_index = list(lr_model.classes_).index(class_name)

coefficients = lr_model.coef_[class_index]
top_indices = coefficients.argsort()[::-1][:15]

for i in top_indices:
    print(feature_names[i], round(coefficients[i], 3))


The strongest features include terms such as `virtual`, `virtual card`, and `work`.

The issue is that phrases like `virtual card` also appear in other virtual card categories, so the model can confuse related intents.


## 8. Model 2: Sentence embeddings and Logistic Regression

The second model uses sentence embeddings instead of TF-IDF.

A sentence embedding represents the full sentence as one dense vector. This can capture overall meaning better when different categories share similar vocabulary.


In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

X_train_embeddings = embedding_model.encode(
    X_train.tolist(),
    batch_size=64,
    show_progress_bar=True
)

X_test_embeddings = embedding_model.encode(
    X_test.tolist(),
    batch_size=64,
    show_progress_bar=True
)

print("Train embeddings:", X_train_embeddings.shape)
print("Test embeddings:", X_test_embeddings.shape)


In [ ]:
embedding_clf = LogisticRegression(max_iter=1000)

embedding_clf.fit(X_train_embeddings, y_train)

y_pred_embed = embedding_clf.predict(X_test_embeddings)

embedding_accuracy = accuracy_score(y_test, y_pred_embed)

print("Embedding Accuracy:", round(embedding_accuracy, 4))


In my run, the sentence embedding model reached about 90.81% accuracy.

This was a clear improvement over the TF-IDF baseline.


In [ ]:
print(classification_report(y_test, y_pred_embed))


For `virtual_card_not_working`, the improvement was especially large.

TF-IDF model:

Precision: 1.00  
Recall: 0.38  
F1: 0.55

Sentence embedding model:

Precision: 1.00  
Recall: 0.80  
F1: 0.89

This suggests that sentence embeddings handled the full meaning of the ticket better when several categories shared similar words.


## 9. Compare F1 score by category

The embedding model improved overall accuracy, but it did not improve every category.

I compare the F1 score for each category to see where embeddings helped and where TF-IDF still performed better.


In [ ]:
tfidf_report = classification_report(
    y_test,
    y_pred_tfidf,
    output_dict=True
)

embed_report = classification_report(
    y_test,
    y_pred_embed,
    output_dict=True
)

comparison = []

for category in y_test.unique():
    tfidf_f1 = tfidf_report[category]["f1-score"]
    embed_f1 = embed_report[category]["f1-score"]

    comparison.append({
        "category": category,
        "tfidf_f1": tfidf_f1,
        "embedding_f1": embed_f1,
        "difference": embed_f1 - tfidf_f1
    })

comparison_df = pd.DataFrame(comparison)


In [ ]:
comparison_df.sort_values(
    "difference",
    ascending=False
).head(15)


In [ ]:
comparison_df.sort_values(
    "difference",
    ascending=True
).head(15)


The comparison shows an important tradeoff.

Sentence embeddings helped most when the classes shared vocabulary but differed in overall meaning.

TF-IDF still performed well for some categories where exact words or phrases were very strong signals.

So embeddings are not automatically better for every class. The right representation depends on the language pattern in the problem.


## 10. Semantic search

The same sentence embeddings can also be used for semantic search.

Instead of assigning one category, semantic search finds historical tickets that are most similar to a new query.

This can help a support agent quickly find related past cases.


In [ ]:
train_texts = train_df["text"].tolist()

ticket_embeddings = embedding_model.encode(
    train_texts,
    batch_size=64,
    show_progress_bar=True
)

print(ticket_embeddings.shape)


In [ ]:
def search_tickets(query, top_n=5):
    query_embedding = embedding_model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        ticket_embeddings
    )[0]

    top_indices = similarities.argsort()[::-1][:top_n]

    results = []

    for i in top_indices:
        results.append({
            "text": train_df["text"].iloc[i],
            "category": train_df["category"].iloc[i],
            "similarity": float(similarities[i])
        })

    return pd.DataFrame(results)


In [ ]:
query = "I was charged twice for the same transaction"

search_tickets(query, top_n=5)


## 11. Evaluate semantic search

For retrieval, I use two simple metrics:

Top 1 accuracy checks whether the nearest historical ticket has the correct category.

Hit at 5 checks whether at least one of the top five retrieved tickets has the correct category.


In [ ]:
test_texts = test_df["text"].tolist()

test_embeddings = embedding_model.encode(
    test_texts,
    batch_size=64,
    show_progress_bar=True
)

similarity_matrix = cosine_similarity(
    test_embeddings,
    ticket_embeddings
)

print(similarity_matrix.shape)


In [ ]:
correct_top1 = 0

for i in range(len(test_df)):
    similarities = similarity_matrix[i]
    best_index = similarities.argmax()

    predicted_category = train_df["category"].iloc[best_index]
    true_category = test_df["category"].iloc[i]

    if predicted_category == true_category:
        correct_top1 += 1

top1_accuracy = correct_top1 / len(test_df)

print("Top 1 Accuracy:", round(top1_accuracy, 4))


In [ ]:
correct_at_5 = 0

for i in range(len(test_df)):
    similarities = similarity_matrix[i]

    top_5_indices = similarities.argsort()[::-1][:5]

    top_5_categories = [
        train_df["category"].iloc[j]
        for j in top_5_indices
    ]

    true_category = test_df["category"].iloc[i]

    if true_category in top_5_categories:
        correct_at_5 += 1

hit_at_5 = correct_at_5 / len(test_df)

print("Hit at 5:", round(hit_at_5, 4))


In my run, semantic search reached about:

Top 1 retrieval accuracy: 92.05%

Hit at 5: 97.01%

This means the correct intent appeared somewhere in the top five retrieved historical tickets for about 97% of test queries.


## 12. Final takeaways

The main lesson from this project was not simply that one model had higher accuracy.

TF-IDF worked well when exact words and phrases clearly separated an intent.

Sentence embeddings worked better when several intents shared similar vocabulary but differed in overall sentence meaning.

Error analysis was especially useful because it showed why the baseline struggled with some classes.

The same sentence embeddings were also useful for semantic search, where the goal was to retrieve similar historical cases instead of assigning a single label.

For a real support system, both approaches could be useful together:

Intent classification for routing tickets

Semantic search for showing agents similar historical cases
